# ДЗ2, опыт23: скорость batch / AMP / bucketing / greedy

Одна **T4**, свежая Colab-сессия. Проверяем скорость; короткие обновления случайных моделей **не оценивают качество обучения**. Старые веса22/100k используются только для greedy-инференса. Beam и sampling отсутствуют.

Загрузить **исходный ZIP22** `hw02-encoder4-decoder4-qwen-bpe8k-swiglu-500k-cosine-tail-75000-to-100000-results.zip` либо **PT22/100k + tokenizer JSON22**. Подходит и `latest.pt`22/100k. Веса/JSON/ZIP сверяются по сохранённым SHA256 **до загрузки данных и моделей**. Другие ноутбуки не нужны.

Выполнить ячейки по порядку. Общий срок —60 минут от первой ячейки, включая подготовку/ожидание загрузки;60 секунд оставлены упаковке. После контроля инференса обучение использует оставшееся время с резервом20 минут вариантам инференса. Бюджетные/OOM/незавершённые строки сохраняются и не участвуют в выборе; вся сетка может не успеть.50+200 шагов не сокращаются. Таймауты проверяются между работами, шагами/чанками; зависший внешний CUDA-драйвер не контролируется Python-дедлайном.

**Если прервали запуск — выполните последнюю ячейку упаковки**, даже если остальные ячейки не завершились. Вернуть один ZIP; notebook с outputs удобен дополнительно. Повторять бенчмарк-ячейки в той же сессии запрещено: свежий запуск — свежая сессия.

Метрика: исполнить `solve()` у reference и генерации; конечные числовые reference, `math.isclose(rel_tol=1e-6, abs_tol=1e-6)`, bool→int. Heldout1024 должен дать991 пригодный reference; контроль22 ожидается189±3. Ручной74 — диагностика ошибок. Полного официального scorer/формата посылки нет; локальный exec-match следует опубликованному принципу gold, AST-ограничения явно учитываются.

In [1]:
import time
if '_BENCH23_STARTED' not in globals():
    _BENCH23_STARTED = time.monotonic()
%pip install -q tokenizers==0.23.2 pyarrow sympy

## 1. Самодостаточные модули и фиксированный протокол

Исполнение программ находится в отдельном импортируемом модуле: процессы `spawn` не наследуют Torch/GPU. AST запрещает опасные имена, атрибуты и импорты; доступны числовые builtins, math и ограниченный числовой SymPy. RLIMIT_CPU и Linux RLIMIT_AS, таймаут программы1с, родительский3с. Это ограниченный запуск числовых программ публичного датасета; заблокированные reference получают отдельный статус. При смерти worker весь пул перезапускается; результаты и порядок сохраняются.

Модели: A4+4/d256/h4/SwiGLU680; B6+6/d384/h6/1024; C6+6/d512/h8/**1368** (≈1365, округление до кратности8). Vocabulary8192/позиции2048/Pre-LN/общий input embedding, untied head, dropout0.1. Точный A совместим с22. Новые модели: AdamW LR1e-4/WD.01/clip1;50 шагов прогрева таймера +200 измеренных, без scheduler. AMP: autocast FP16 +GradScaler, unscale→clip→step. Инициализация/батчи зафиксированы; пары FP32/AMP получают одинаковые планы. Batch — пары, без gradient accumulation.

20 конфигураций: контрольA/FP32/batch8/random.sample/обычныйAdamW; A/AMP/batch8 с тем же планом;18 сочетаний A/B/C×FP32/AMP×32/128/256, bucketing+fusedAdamW при наличии. В чанках4096 случайных пар сортируем по сумме длин source/target, формируем батчи и перемешиваем их. Матрица меняет сразу batch/bucketing/fused относительно контроля; выигрыш отдельной причины по ней не доказывается.

Выбор train: максимум **полезных target-токенов/с**, оба memory peaks<14×10^9 байт,200 полных шагов, конечные loss/градиенты, без AMP-пропусков на измерении. Также печатаются source и сумма токенов, пары/с, padding и победители каждой модели. Это выбор скорости, не архитектуры по качеству.

In [2]:
import os,sys,json,hashlib,random,gc,subprocess,zipfile,shutil,math
from pathlib import Path
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['OMP_NUM_THREADS']='2'
os.environ['OPENBLAS_NUM_THREADS']='1'
MODULE_DIR=Path('hw02-bench23-modules')
MODULE_DIR.mkdir(exist_ok=True)

MODULE_SOURCES={'bench23_core.py': 'import ast,contextlib,gc,hashlib,json,math,random,time\nfrom array import array\nfrom pathlib import Path\nfrom collections import Counter\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.nn.utils.rnn import pad_sequence\nfrom tokenizers import Tokenizer\n\nPAD,BOS,EOS,TOKEN_OFFSET=0,1,2,3\nVOCAB=8192\nMAX_POSITIONS=2048\nGENERATION_CAP=512\nSEED=42\nMODEL_SPECS={\n \'A\':dict(d_model=256,heads=4,encoder_layers=4,decoder_layers=4,swiglu_dim=680),\n \'B\':dict(d_model=384,heads=6,encoder_layers=6,decoder_layers=6,swiglu_dim=1024),\n \'C\':dict(d_model=512,heads=8,encoder_layers=6,decoder_layers=6,swiglu_dim=1368)}\n\nclass SwiGLUInput(nn.Module):\n def __init__(self,width,hidden):\n  super().__init__();self.gate=nn.Linear(width,hidden);self.value=nn.Linear(width,hidden)\n def forward(self,x):return F.silu(self.gate(x))*self.value(x)\n\ndef replace_ffn_with_swiglu(transformer,width,hidden):\n with torch.random.fork_rng(devices=[]):\n  torch.set_rng_state(torch.Generator().manual_seed(SEED).get_state())\n  for layer in list(transformer.encoder.layers)+list(transformer.decoder.layers):\n   layer.linear1=SwiGLUInput(width,hidden);layer.linear2=nn.Linear(hidden,width);layer.activation=nn.Identity()\n   if isinstance(layer,nn.TransformerEncoderLayer):layer.activation_relu_or_gelu=0\n   for module in [layer.linear1.gate,layer.linear1.value,layer.linear2]:nn.init.xavier_uniform_(module.weight)\n\nclass SimpleTransformer(nn.Module):\n def __init__(self,d_model=256,heads=4,encoder_layers=4,decoder_layers=4,swiglu_dim=680,\n              vocab=VOCAB,max_positions=MAX_POSITIONS):\n  super().__init__();self.width=d_model;self.vocab=vocab\n  self.tokens=nn.Embedding(vocab,d_model,padding_idx=PAD);self.positions=nn.Embedding(max_positions,d_model)\n  self.transformer=nn.Transformer(d_model=d_model,nhead=heads,num_encoder_layers=encoder_layers,\n   num_decoder_layers=decoder_layers,dim_feedforward=4*d_model,dropout=.1,batch_first=True,norm_first=True)\n  self.transformer.encoder.enable_nested_tensor=False;self.transformer.encoder.use_nested_tensor=False\n  self.head=nn.Linear(d_model,vocab);replace_ffn_with_swiglu(self.transformer,d_model,swiglu_dim)\n def embed(self,ids):\n  if ids.shape[1]>self.positions.num_embeddings:raise ValueError(\'Position overflow\')\n  return self.tokens(ids)*math.sqrt(self.width)+self.positions(torch.arange(ids.shape[1],device=ids.device))\n def decode_prefix(self,tgt,memory,source_padding):\n  mask=torch.ones(tgt.shape[1],tgt.shape[1],device=tgt.device,dtype=torch.bool).triu(1)\n  return self.transformer.decoder(self.embed(tgt),memory,tgt_mask=mask,tgt_key_padding_mask=tgt.eq(PAD),memory_key_padding_mask=source_padding)\n def forward(self,src,tgt):\n  memory=self.transformer.encoder(self.embed(src),src_key_padding_mask=src.eq(PAD))\n  return self.head(self.decode_prefix(tgt,memory,src.eq(PAD)))\n @torch.inference_mode()\n def generate(self,src,max_new_tokens=512):\n  self.eval();memory=self.transformer.encoder(self.embed(src),src_key_padding_mask=src.eq(PAD))\n  tgt=torch.full((len(src),1),BOS,device=src.device,dtype=torch.long);done=torch.zeros(len(src),device=src.device,dtype=torch.bool)\n  for _ in range(max_new_tokens):\n   logits=self.head(self.decode_prefix(tgt,memory,src.eq(PAD))[:,-1]);logits[:,[PAD,BOS]]=-torch.inf\n   token=logits.argmax(-1).masked_fill(done,PAD);done|=token.eq(EOS);tgt=torch.cat([tgt,token[:,None]],dim=1)\n   if bool(done.all()):break\n  return tgt[:,1:]\n\ndef sync(device):\n if device.type==\'cuda\':torch.cuda.synchronize(device)\n\ndef init_tokenizer(path):\n tokenizer=Tokenizer.from_file(str(path))\n if tokenizer.get_vocab_size()!=VOCAB-TOKEN_OFFSET:raise ValueError(\'Unexpected vocabulary\')\n visible=list(range(33,127))+list(range(161,173))+list(range(174,256));bchars={b:chr(b)for b in visible}\n for i,b in enumerate(b for b in range(256)if b not in bchars):bchars[b]=chr(256+i)\n charbytes={c:b for b,c in bchars.items()}\n tokenbytes=[b\'\',b\'\',b\'\']+[bytes(charbytes[c]for c in tokenizer.id_to_token(i))for i in range(VOCAB-3)]\n return tokenizer,tokenbytes\n\ndef encode(tokenizer,text):return [i+TOKEN_OFFSET for i in tokenizer.encode(text,add_special_tokens=False).ids]\ndef decode_bytes(tokenbytes,ids):\n chunks=[]\n for i in ids:\n  if i==EOS:break\n  if i>=TOKEN_OFFSET:chunks.append(tokenbytes[i])\n return b\'\'.join(chunks)\ndef source_batch(tokenizer,questions,device):\n return pad_sequence([torch.tensor([BOS]+encode(tokenizer,q)+[EOS])for q in questions],batch_first=True,padding_value=PAD).to(device)\ndef collate_cache(cache,indices,device):\n src=pad_sequence([torch.tensor(cache[i][0],dtype=torch.long)for i in indices],batch_first=True,padding_value=PAD)\n tgt=pad_sequence([torch.tensor([BOS]+list(cache[i][1]),dtype=torch.long)for i in indices],batch_first=True,padding_value=PAD)\n labels=pad_sequence([torch.tensor(list(cache[i][1])+[EOS],dtype=torch.long)for i in indices],batch_first=True,padding_value=PAD)\n counts={\'source_tokens\':sum(len(cache[i][0])for i in indices),\'target_tokens\':sum(len(cache[i][1])+1 for i in indices),\n  \'source_slots\':src.numel(),\'target_slots\':labels.numel(),\'pairs\':len(indices)}\n return src.to(device),tgt.to(device),labels.to(device),counts\n\ndef random_plan(n,batch,steps,seed=230042):\n rng=random.Random(seed);return [rng.sample(range(n),batch)for _ in range(steps)]\ndef bucket_plan(cache,batch,steps,seed=230042,chunk_pairs=4096):\n if chunk_pairs%batch:raise ValueError(\'Chunk must be divisible by batch\')\n rng=random.Random(seed);plans=[]\n while len(plans)<steps:\n  chunk=rng.sample(range(len(cache)),min(chunk_pairs,len(cache)))\n  chunk.sort(key=lambda i:(len(cache[i][0])+len(cache[i][1])+1,len(cache[i][0]),len(cache[i][1])))\n  batches=[chunk[i:i+batch]for i in range(0,len(chunk)-batch+1,batch)]\n  if not batches:raise ValueError(\'Too few data for this batch size\')\n  rng.shuffle(batches);plans.extend(batches)\n return plans[:steps]\n\ndef sdpa_flags():\n return {k:bool(getattr(torch.backends.cuda,k+\'_sdp_enabled\')())for k in [\'flash\',\'mem_efficient\',\'math\',\'cudnn\']if hasattr(torch.backends.cuda,k+\'_sdp_enabled\')}\ndef profile_sdpa(call,device):\n if device.type!=\'cuda\':return {\'status\':\'CPU_smoke\',\'operators\':[]}\n with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU,torch.profiler.ProfilerActivity.CUDA])as prof:\n  call();sync(device)\n ops=sorted({e.key for e in prof.key_averages()if \'scaled_dot_product\'in e.key})\n return {\'status\':\'observed\',\'operators\':ops,\'scope\':\'one warmup forward/backward for this shape/dtype, outside measured steps; backend may change with shapes\'}\n\ndef fused_adamw_available(device):\n if device.type!=\'cuda\':return False\n try:\n  p=nn.Parameter(torch.ones(1,device=device));opt=torch.optim.AdamW([p],lr=1e-4,fused=True);p.grad=torch.ones_like(p);opt.step();sync(device)\n  del opt,p;return True\n except (TypeError,RuntimeError):return False\n\ndef benchmark_train(spec,cache,plan,precision,device,warmup=50,measured=200,fused=False,\n                    deadline=None,profile=False,seed=230042):\n """Fresh scratch weights, fixed LR1e-4. Warmup is timing warmup, not LR warmup."""\n summary={\'model_spec\':spec,\'precision\':precision,\'batch_pairs\':len(plan[0]),\'warmup_planned\':warmup,\n  \'measured_planned\':measured,\'fused_adamw\':fused,\'status\':\'started\',\'completed_warmup\':0,\'completed_measured\':0}\n if len(plan)<warmup+measured:raise ValueError(\'Plan too short\')\n model=opt=scaler=None\n try:\n  gc.collect()\n  if device.type==\'cuda\':torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats(device)\n  torch.manual_seed(seed)\n  if device.type==\'cuda\':torch.cuda.manual_seed_all(seed)\n  model=SimpleTransformer(**spec).to(device);model.train();summary[\'parameters\']=sum(p.numel()for p in model.parameters())\n  opt=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=.01,**({\'fused\':True}if fused else{}))\n  amp=precision==\'amp_fp16\';scaler=torch.amp.GradScaler(device.type,enabled=amp,init_scale=65536)\n  finite_losses=torch.empty(warmup+measured,dtype=torch.bool,device=device)\n  finite_grads=torch.empty_like(finite_losses)\n  amp_found_inf=torch.zeros(warmup+measured,dtype=torch.float32,device=device)\n  if amp and not hasattr(scaler,\'_found_inf_per_device\'):raise RuntimeError(\'GradScaler diagnostic API unavailable: cannot count skips reliably\')\n  counts=Counter();measured_seconds=0.;warmup_started=time.monotonic()\n  def step(i,collect):\n   src,tgt,labels,batch_counts=collate_cache(cache,plan[i],device)\n   opt.zero_grad(set_to_none=True)\n   with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=amp):\n    logits=model(src,tgt);loss=F.cross_entropy(logits.reshape(-1,model.vocab),labels.reshape(-1),ignore_index=PAD)\n   if amp:\n    scaler.scale(loss).backward();scaler.unscale_(opt)\n    # Diagnostics only: one-device found_inf recorded on GPU, read after timer.\n    # Unlike grad-norm overflow or optimizer hooks with fusedAdamW, this is\n    # the exact flag GradScaler supplies to the optimizer to skip this update.\n    found=scaler._found_inf_per_device(opt)\n    if len(found)!=1:raise RuntimeError(\'Expected one-device GradScaler found_inf\')\n    amp_found_inf[i]=next(iter(found.values()))\n   else:loss.backward()\n   norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n   finite_losses[i]=torch.isfinite(loss.detach());finite_grads[i]=torch.isfinite(norm.detach())\n   if amp:scaler.step(opt);scaler.update()\n   else:opt.step()\n   if collect:counts.update(batch_counts)\n  for i in range(warmup):\n   if deadline is not None and time.monotonic()>=deadline:summary[\'status\']=\'budget_warmup\';return summary\n   if profile and i==0:summary[\'sdpa\']=profile_sdpa(lambda:step(i,False),device)\n   else:step(i,False)\n   summary[\'completed_warmup\']=i+1\n   if (i+1)%10==0:sync(device)\n  sync(device);warmup_seconds=time.monotonic()-warmup_started\n  summary[\'warmup_seconds\']=warmup_seconds\n  if deadline is not None and time.monotonic()+warmup_seconds/warmup*measured*1.15>=deadline:\n   summary[\'status\']=\'budget_projection\';return summary\n  # Chunk boundary sync caps outstanding GPU work and provides safe deadline checks.\n  # Collation, H2D, loss/backward/clip/AdamW/scaler and diagnostics included.\n  for lo in range(0,measured,20):\n   if deadline is not None and time.monotonic()>=deadline:summary[\'status\']=\'budget_measured\';return summary\n   sync(device);start=time.perf_counter()\n   for offset in range(lo,min(lo+20,measured)):step(warmup+offset,True)\n   sync(device);measured_seconds+=time.perf_counter()-start\n   summary[\'completed_measured\']=min(lo+20,measured)\n  good_loss=bool(finite_losses[warmup:].all());good_grad=bool(finite_grads[warmup:].all())\n  skipped=int(amp_found_inf[warmup:].ne(0).sum().item())if amp else 0\n  summary.update(status=\'ok\'if good_loss and good_grad else\'non_finite\',seconds=measured_seconds,\n   ms_per_step=1000*measured_seconds/measured,pairs_per_second=counts[\'pairs\']/measured_seconds,\n   useful_target_tokens_per_second=counts[\'target_tokens\']/measured_seconds,\n   useful_source_tokens_per_second=counts[\'source_tokens\']/measured_seconds,\n   useful_source_plus_target_tokens_per_second=(counts[\'source_tokens\']+counts[\'target_tokens\'])/measured_seconds,\n   counts=dict(counts),source_padding_fraction=1-counts[\'source_tokens\']/counts[\'source_slots\'],\n   target_padding_fraction=1-counts[\'target_tokens\']/counts[\'target_slots\'],\n   total_padding_fraction=1-(counts[\'source_tokens\']+counts[\'target_tokens\'])/(counts[\'source_slots\']+counts[\'target_slots\']),\n   all_warmup_losses_finite=bool(finite_losses[:warmup].all()),all_measured_losses_finite=good_loss,all_measured_gradient_norms_finite=good_grad,\n   warmup_AMP_skipped_updates=int(amp_found_inf[:warmup].ne(0).sum().item())if amp else 0,\n   measured_nonfinite_gradient_norm_steps=int((~finite_grads[warmup:]).sum().item()),\n   AMP_skip_diagnostic=\'GradScaler one-device found_inf, stored on device beforeclip, host read outside timer\'if amp else None,\n   measured_AMP_skipped_updates=skipped,final_scale=scaler.get_scale()if amp else None,\n   peak_allocated_bytes=torch.cuda.max_memory_allocated(device)if device.type==\'cuda\'else None,\n   peak_reserved_bytes=torch.cuda.max_memory_reserved(device)if device.type==\'cuda\'else None)\n  return summary\n except torch.cuda.OutOfMemoryError as e:\n  summary.update(status=\'oom\',error=str(e)[:400]);return summary\n except RuntimeError as e:\n  summary.update(status=\'oom\'if \'out of memory\'in str(e).lower()else\'error\',error=str(e)[:400]);return summary\n finally:\n  if device.type==\'cuda\':\n   summary.setdefault(\'peak_allocated_bytes\',torch.cuda.max_memory_allocated(device))\n   summary.setdefault(\'peak_reserved_bytes\',torch.cuda.max_memory_reserved(device))\n  del model,opt,scaler;gc.collect()\n  if device.type==\'cuda\':torch.cuda.empty_cache()\n\n\nclass CachedDecoder:\n    @torch.inference_mode()\n    def __init__(self, model, src, capacity):\n        model.eval()\n        if not 0 < capacity <= model.positions.num_embeddings:\n            raise ValueError(\'Invalid cache capacity\')\n        self.model, self.capacity, self.position = model, capacity, 0\n        self.batch_size = len(src)\n        self.width = model.tokens.embedding_dim\n        self.heads = model.transformer.decoder.layers[0].self_attn.num_heads\n        self.head_dim = self.width // self.heads\n        memory = model.transformer.encoder(model.embed(src), src_key_padding_mask=src.eq(PAD))\n        # SDPA bool masks use True for allowed keys (opposite to MHA padding masks).\n        self.source_allowed = src.ne(PAD)[:, None, None, :]\n        self.valid = torch.zeros(len(src), capacity, dtype=torch.bool, device=src.device)\n        self.position_vectors = model.positions(torch.arange(capacity, device=src.device))\n        self.layers = []\n        for layer in model.transformer.decoder.layers:\n            if not layer.norm_first:\n                raise ValueError(\'This experiment requires Pre-LN decoder layers\')\n            for attention in (layer.self_attn, layer.multihead_attn):\n                if (not attention._qkv_same_embed_dim or attention.num_heads != self.heads\n                        or attention.bias_k is not None or attention.bias_v is not None\n                        or attention.add_zero_attn):\n                    raise ValueError(\'Unsupported attention configuration\')\n            attention = layer.multihead_attn\n            projected_kv = F.linear(memory, attention.in_proj_weight[self.width:],\n                                    attention.in_proj_bias[self.width:])\n            cross_k, cross_v = projected_kv.chunk(2, dim=-1)\n            shape = (len(src), self.heads, capacity, self.head_dim)\n            self.layers.append({\n                \'self_k\': memory.new_empty(shape), \'self_v\': memory.new_empty(shape),\n                \'cross_k\': self.split_heads(cross_k), \'cross_v\': self.split_heads(cross_v),\n            })\n\n    def split_heads(self, values):\n        return values.reshape(self.batch_size, -1, self.heads, self.head_dim).transpose(1, 2)\n\n    def attend(self, attention, q, k, v, allowed):\n        # One query at the latest position; all supplied keys are from its past/current.\n        # is_causal=True would use an upper-left 1 x T mask and incorrectly hide the past.\n        output = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed,\n                                                dropout_p=0.0, is_causal=False)\n        output = output.transpose(1, 2).contiguous().reshape(self.batch_size, 1, self.width)\n        return attention.out_proj(output)\n\n    @torch.inference_mode()\n    def step(self, token_ids):\n        if self.position >= self.capacity:\n            raise ValueError(\'Cache capacity exceeded\')\n        if token_ids.ndim != 1 or len(token_ids) != self.batch_size:\n            raise ValueError(\'One token per batch row is required\')\n        t = self.position\n        x = self.model.tokens(token_ids[:, None]) * math.sqrt(self.width)\n        x = x + self.position_vectors[t][None, None, :]\n        self.valid[:, t] = token_ids.ne(PAD)\n        self_allowed = self.valid[:, None, None, :t + 1]\n        for layer, cache in zip(self.model.transformer.decoder.layers, self.layers):\n            attention = layer.self_attn\n            q, k, v = F.linear(layer.norm1(x), attention.in_proj_weight, attention.in_proj_bias).chunk(3, dim=-1)\n            cache[\'self_k\'][:, :, t:t + 1] = self.split_heads(k)\n            cache[\'self_v\'][:, :, t:t + 1] = self.split_heads(v)\n            self_output = self.attend(attention, self.split_heads(q),\n                cache[\'self_k\'][:, :, :t + 1], cache[\'self_v\'][:, :, :t + 1], self_allowed)\n            x = x + layer.dropout1(self_output)\n\n            attention = layer.multihead_attn\n            q = F.linear(layer.norm2(x), attention.in_proj_weight[:self.width], attention.in_proj_bias[:self.width])\n            cross_output = self.attend(attention, self.split_heads(q),\n                                      cache[\'cross_k\'], cache[\'cross_v\'], self.source_allowed)\n            x = x + layer.dropout2(cross_output)\n            feed_forward = layer.linear2(layer.dropout(layer.activation(layer.linear1(layer.norm3(x)))))\n            x = x + layer.dropout3(feed_forward)\n\n        if self.model.transformer.decoder.norm is not None:\n            x = self.model.transformer.decoder.norm(x)\n        self.position += 1\n        return self.model.head(x[:, 0])\n\n\ndef code_properties(code):\n    def scoped_nodes(node):\n        stack = [node]\n        while stack:\n            current = stack.pop()\n            yield current\n            if not isinstance(current, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Lambda)):\n                stack.extend(ast.iter_child_nodes(current))\n    try:\n        tree = ast.parse(code)\n        functions = [n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == \'solve\']\n        has_return = any(isinstance(n, ast.Return) and n.value is not None\n            for f in functions for stmt in f.body for n in scoped_nodes(stmt))\n        return {\'parses_as_python\': True, \'defines_solve\': bool(functions), \'solve_returns_value\': has_return,\n                \'syntax_error_type\': None, \'syntax_error_message\': None, \'indentation_error\': False}\n    except (SyntaxError, ValueError, RecursionError) as error:\n        return {\'parses_as_python\': False, \'defines_solve\': False, \'solve_returns_value\': False,\n                \'syntax_error_type\': type(error).__name__, \'syntax_error_message\': str(error),\n                \'indentation_error\': isinstance(error, IndentationError)}\n\nclass BudgetExpired(Exception):pass\n\ndef check_deadline(deadline):\n if deadline is not None and time.monotonic()>=deadline:raise BudgetExpired(\'60 minute run budget / phase reservation\')\n\ndef active_tokens(ids):\n ids=[int(i)for i in ids if i!=PAD]\n return ids[:ids.index(EOS)+1]if EOS in ids else ids\n\n@torch.inference_mode()\ndef generate_control(model,src,cap=GENERATION_CAP,deadline=None):\n """The 21/22 greedy path: growing output via cat; done.all CPU check each step."""\n cache=CachedDecoder(model,src,cap)\n tgt=torch.full((len(src),1),BOS,device=src.device,dtype=torch.long)\n done=torch.zeros(len(src),device=src.device,dtype=torch.bool)\n output_bytes=torch.zeros(len(src),device=src.device,dtype=torch.long)\n byte_stopped=done.clone()\n for step in range(cap):\n  if step%16==0:check_deadline(deadline)\n  logits=cache.step(tgt[:,-1]);logits[:,[PAD,BOS]]=-torch.inf\n  token=logits.argmax(-1).masked_fill(done,PAD)\n  if hasattr(model,\'token_byte_lengths\'):output_bytes+=model.token_byte_lengths[token]\n  done|=token.eq(EOS)\n  tgt=torch.cat([tgt,token[:,None]],dim=1)\n  if bool(done.all()):break\n return tgt[:,1:],cache.position\n\n@torch.inference_mode()\ndef generate_preallocated(model,src,cap=GENERATION_CAP,deadline=None,check_every=16):\n """Only output storage and termination checks change; same encoder/KV/argmax."""\n cache=CachedDecoder(model,src,cap)\n output=torch.full((len(src),cap),PAD,device=src.device,dtype=torch.long)\n token=torch.full((len(src),),BOS,device=src.device,dtype=torch.long)\n done=torch.zeros(len(src),device=src.device,dtype=torch.bool)\n for step in range(cap):\n  if step%check_every==0:check_deadline(deadline)\n  logits=cache.step(token);logits[:,[PAD,BOS]]=-torch.inf\n  token=logits.argmax(-1).masked_fill(done,PAD);output[:,step]=token;done|=token.eq(EOS)\n  # Only one device->CPU synchronization per 16 steps, plus final output transfer.\n  # Rows already at EOS remain PAD; active_tokens removes these extra slots.\n  if (step+1)%check_every==0 and bool(done.all()):break\n return output[:,:cache.position],cache.position\n\ndef restore_order(ordered_values,original_indices,size):\n if len(ordered_values)!=size or sorted(original_indices)!=list(range(size)):\n  raise ValueError(\'Not a complete permutation\')\n restored=[None]*size\n for value,i in zip(ordered_values,original_indices):restored[i]=value\n return restored\n\ndef inspect_output(row,ids,tokenizer,tokenbytes):\n active=active_tokens(ids);raw=decode_bytes(tokenbytes,active)\n try:code=raw.decode(\'utf-8\');valid=True\n except UnicodeDecodeError:code=raw.decode(\'utf-8\',errors=\'replace\');valid=False\n example={\'question\':row[\'question\'],\'generated_code\':code,\'generated_token_ids\':active,\n  \'generated_tokens_including_eos\':len(active),\'generated_utf8_bytes\':len(raw),\'valid_utf8\':valid,\n  \'eos_emitted\':EOS in active,\'reached_token_cap\':len(active)==GENERATION_CAP,\n  \'token_cap_without_eos\':len(active)==GENERATION_CAP and EOS not in active,\n  \'byte_budget_stop\':False,\'stop_reason\':\'eos\'if EOS in active else\'token_cap\',**code_properties(code)}\n if \'code\'in row:\n  gold=encode(tokenizer,row[\'code\'])+[EOS]\n  example.update(reference_code=row[\'code\'],reference_utf8_bytes=len(row[\'code\'].encode()),\n   reference_bpe_tokens_including_eos=len(gold),exact_code_match=valid and code==row[\'code\'],exact_tokens_with_eos=active==gold)\n return example\n\ndef sha_file(path):\n h=hashlib.sha256()\n with Path(path).open(\'rb\')as f:\n  for chunk in iter(lambda:f.read(1024**2),b\'\'):h.update(chunk)\n return h.hexdigest()\n\ndef write_json(path,value):\n path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_suffix(path.suffix+\'.tmp\')\n tmp.write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False),encoding=\'utf-8\');tmp.replace(path)\n\ndef run_inference_split(checkpoint,tokenizer_path,rows,split,variant,device,outdir,deadline=None,baseline=None,cap=GENERATION_CAP,profile=False):\n """Full wall starts BEFORE torch.load; ends after answers file. Exec is outside it."""\n summary={\'split\':split,**variant,\'status\':\'started\',\'examples_planned\':len(rows),\'weight_loads\':1}\n model=state=None\n try:\n  check_deadline(deadline);gc.collect()\n  if device.type==\'cuda\':torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats(device)\n  full_started=time.perf_counter()\n  # Only the exact hashed checkpoint is allowed; it contains legacy Python/RNG metadata.\n  state=torch.load(checkpoint,map_location=\'cpu\',weights_only=False)\n  if state[\'steps\']!=100000:raise ValueError(\'Requires22/100k weights\')\n  tokenizer,tokenbytes=init_tokenizer(tokenizer_path)\n  model=SimpleTransformer(**MODEL_SPECS[\'A\']);model.load_state_dict(state[\'model\'],strict=True);del state;state=None\n  model=model.to(device);model.eval()\n  model.token_byte_lengths=torch.tensor([len(b)for b in tokenbytes],device=device)\n  if variant[\'precision\']==\'fp16\':model.half()\n  generate=generate_control if variant[\'path\']==\'control\'else generate_preallocated\n  warmup=lambda:generate(model,source_batch(tokenizer,[rows[0][\'question\']],device),cap=min(8,cap),deadline=deadline)\n  if profile:summary[\'sdpa\']=profile_sdpa(warmup,device)\n  else:warmup()\n  sync(device);started=time.perf_counter()\n  # Encoding/sorting is included in the historical timer region.\n  if variant[\'path\']==\'control\':indices=list(range(len(rows)));encoded=None\n  else:\n   encoded=[[BOS]+encode(tokenizer,r[\'question\'])+[EOS]for r in rows]\n   indices=sorted(range(len(rows)),key=lambda i:(len(encoded[i]),i))\n  ordered=[];decode_steps=0;decode_row_steps=0\n  for lo in range(0,len(rows),variant[\'batch_pairs\']):\n   check_deadline(deadline);batch_indices=indices[lo:lo+variant[\'batch_pairs\']]\n   if encoded is None:src=source_batch(tokenizer,[rows[i][\'question\']for i in batch_indices],device)\n   else:src=pad_sequence([torch.tensor(encoded[i],dtype=torch.long)for i in batch_indices],batch_first=True,padding_value=PAD).to(device)\n   tokens,steps=generate(model,src,cap=cap,deadline=deadline)\n   decode_steps+=steps;decode_row_steps+=steps*len(batch_indices)\n   ordered.extend(inspect_output(rows[i],ids,tokenizer,tokenbytes)for i,ids in zip(batch_indices,tokens.cpu().tolist()))\n  examples=restore_order(ordered,indices,len(rows));sync(device);seconds=time.perf_counter()-started\n  outdir=Path(outdir);outdir.mkdir(parents=True,exist_ok=True)\n  answers=outdir/f\'{variant["id"]}-{split}-answers.jsonl\'\n  # Neutral local format; the official bot format has not been supplied.\n  with answers.open(\'w\',encoding=\'utf-8\')as f:\n   for i,e in enumerate(examples):f.write(json.dumps({\'id\':i,\'question\':e[\'question\'],\'code\':e[\'generated_code\']},ensure_ascii=False)+\'\\n\')\n  full_seconds=time.perf_counter()-full_started\n  if baseline is not None:\n   if len(baseline)!=len(examples)or any(a[\'question\']!=b[\'question\']for a,b in zip(examples,baseline)):raise ValueError(\'Control order mismatch\')\n   identical=sum(a[\'generated_token_ids\']==b[\'generated_token_ids\']for a,b in zip(examples,baseline))\n  else:identical=len(examples)\n  summary.update(status=\'ok\',examples=len(examples),seconds_historical_scope=seconds,seconds_full_wall=full_seconds,\n   answers_per_second=len(examples)/seconds,decode_steps_sum=decode_steps,decode_row_steps_sum=decode_row_steps,\n   generated_tokens_including_eos=sum(e[\'generated_tokens_including_eos\']for e in examples),\n   token_identical_count=identical,token_identical_fraction=identical/len(examples),answer_file=answers.name,\n   peak_allocated_bytes=torch.cuda.max_memory_allocated(device)if device.type==\'cuda\'else None,\n   peak_reserved_bytes=torch.cuda.max_memory_reserved(device)if device.type==\'cuda\'else None,\n   historical_timer=\'encode/sort, H2D, encoder/KV decode, CPU token/UTF8 decoding, AST and reference-token diagnostics; excludes warmup/load/exec/file\',\n   full_timer=\'torch.load, tokenizer restore, strict model restore/cast/H2D, warmup, generation/diagnostics, order restore and answers write; excludes download/cache/exec scoring\')\n  return summary,examples\n except torch.cuda.OutOfMemoryError as e:\n  summary.update(status=\'oom\',error=str(e)[:400]);return summary,None\n except BudgetExpired as e:\n  summary.update(status=\'budget\',error=str(e));return summary,None\n except RuntimeError as e:\n  summary.update(status=\'oom\'if \'out of memory\'in str(e).lower()else\'error\',error=str(e)[:400]);return summary,None\n finally:\n  if device.type==\'cuda\':\n   summary.setdefault(\'peak_allocated_bytes\',torch.cuda.max_memory_allocated(device))\n   summary.setdefault(\'peak_reserved_bytes\',torch.cuda.max_memory_reserved(device))\n  del model,state;gc.collect()\n  if device.type==\'cuda\':torch.cuda.empty_cache()\n\ndef select_train(rows,memory_limit=14_000_000_000):\n valid=[r for r in rows if r.get(\'status\')==\'ok\'and max(r.get(\'peak_allocated_bytes\')or 0,r.get(\'peak_reserved_bytes\')or 0)<memory_limit\n  and r.get(\'completed_measured\')==200 and r.get(\'all_measured_losses_finite\')and r.get(\'all_measured_gradient_norms_finite\')\n  and r.get(\'measured_AMP_skipped_updates\',0)==0]\n return max(valid,key=lambda r:r[\'useful_target_tokens_per_second\'])if valid else None\n\ndef select_inference(rows,control_matches,denominator=991,max_drop=.005):\n valid=[r for r in rows if r.get(\'status\')==\'ok\'and r[\'heldout\'][\'exec_match\'][\'denominator\']==denominator\n  and r[\'heldout\'][\'exec_match\'][\'matches\']/denominator>=control_matches/denominator-max_drop]\n return min(valid,key=lambda r:r[\'test\'][\'seconds_full_wall\'])if valid else None\n', 'bench23_exec.py': '"""Bounded numeric-program execution for this public dataset; no arbitrary imports/I/O."""\nimport ast\nimport builtins\nimport math\nimport multiprocessing as mp\nimport os\nimport resource\nimport signal\nimport sys\nimport time\nfrom collections import Counter, deque\nfrom types import SimpleNamespace\n\nSAFE_BUILTINS = {n: getattr(builtins, n) for n in (\n    \'abs\',\'all\',\'any\',\'bool\',\'dict\',\'divmod\',\'enumerate\',\'float\',\'int\',\'len\',\'list\',\n    \'max\',\'min\',\'pow\',\'range\',\'reversed\',\'round\',\'set\',\'sorted\',\'str\',\'sum\',\'tuple\',\'zip\',\n    \'Exception\',\'ValueError\',\'TypeError\',\'ZeroDivisionError\',\'OverflowError\',\'ArithmeticError\')}\nFORBIDDEN_NAMES = {\'open\',\'input\',\'eval\',\'exec\',\'compile\',\'globals\',\'locals\',\'vars\',\'getattr\',\n    \'setattr\',\'delattr\',\'dir\',\'help\',\'breakpoint\',\'exit\',\'quit\',\'os\',\'sys\',\'subprocess\',\n    \'socket\',\'ctypes\',\'importlib\',\'builtins\',\'resource\',\'signal\',\'pickle\',\'marshal\',\n    \'memoryview\',\'type\',\'object\',\'super\'}\nFORBIDDEN_ATTRS = {\'mro\',\'subclasses\',\'format_map\',\'gi_frame\',\'f_globals\',\'f_locals\',\n    \'tb_frame\',\'func_globals\',\'code\',\'modules\',\'system\',\'popen\',\'load\',\'loads\',\'dump\',\'dumps\'}\nSAFE_BUILTINS[\'print\']=lambda *args,**kwargs:None  # no program output or file handle\nALLOWED_IMPORTS = {\'math\',\'sympy\'}\nSYM_NAMES = {\'Symbol\',\'symbols\',\'Rational\',\'Integer\',\'Float\',\'Eq\',\'solve\',\'sqrt\',\'simplify\'}\nMAX_CODE_BYTES=100000\nMAX_AST_NODES=5000\n\nclass ProgramTimeout(Exception): pass\n\ndef filter_program(code):\n    if not isinstance(code,str) or len(code.encode(\'utf-8\'))>MAX_CODE_BYTES:\n        return None,\'blocked_size\'\n    try: tree=ast.parse(code)\n    except (SyntaxError,ValueError,RecursionError): return None,\'syntax_error\'\n    nodes=list(ast.walk(tree))\n    if len(nodes)>MAX_AST_NODES:return None,\'blocked_size\'\n    for n in nodes:\n        if isinstance(n,ast.Name) and (n.id.startswith(\'__\') or n.id in FORBIDDEN_NAMES):\n            return None,\'blocked_name\'\n        if isinstance(n,ast.Attribute) and (n.attr.startswith(\'_\') or n.attr in FORBIDDEN_ATTRS):\n            return None,\'blocked_attribute\'\n        if isinstance(n,(ast.ClassDef,ast.AsyncFunctionDef,ast.Await,ast.Yield,ast.YieldFrom)):\n            return None,\'blocked_construct\'\n        if isinstance(n,(ast.FunctionDef,ast.arg)):\n            name=n.name if isinstance(n,ast.FunctionDef) else n.arg\n            if name.startswith(\'__\') or name in FORBIDDEN_NAMES:return None,\'blocked_name\'\n        if isinstance(n,ast.Import):\n            if any(a.name not in ALLOWED_IMPORTS or (a.asname and a.asname.startswith(\'_\'))for a in n.names):\n                return None,\'blocked_import\'\n        if isinstance(n,ast.ImportFrom):\n            if n.level or n.module not in ALLOWED_IMPORTS:return None,\'blocked_import\'\n            if any(a.name.startswith(\'_\') or a.name==\'*\'or (a.asname and a.asname.startswith(\'_\'))for a in n.names):\n                return None,\'blocked_import\'\n            if n.module==\'sympy\' and any(a.name not in SYM_NAMES for a in n.names):return None,\'blocked_import\'\n    return tree,None\n\ndef safe_sympy():\n    import sympy\n    def no_strings(x):\n        if isinstance(x,str):raise ValueError(\'String expressions forbidden\')\n        if isinstance(x,(tuple,list)):\n            for y in x:no_strings(y)\n        if isinstance(x,dict):\n            for k,v in x.items():no_strings(k);no_strings(v)\n    def wrapped(function):\n        def call(*args,**kwargs):\n            no_strings(args);no_strings(list(kwargs.values()))\n            return function(*args,**kwargs)\n        return call\n    def symbol(name,*args,**kwargs):\n        if not isinstance(name,str) or not name or not name.replace(\'_\',\'\').isalnum():raise ValueError(\'Bad symbol\')\n        return sympy.Symbol(name,*args,**kwargs)\n    def symbols(names,*args,**kwargs):\n        if not isinstance(names,str)or any(not(c.isalnum()or c in \' _,:\')for c in names):raise ValueError(\'Bad symbols\')\n        return sympy.symbols(names,*args,**kwargs)\n    return SimpleNamespace(Symbol=symbol,symbols=symbols,\n        **{n:wrapped(getattr(sympy,n))for n in SYM_NAMES-{\'Symbol\',\'symbols\'}})\n\n_MODULES={}\ndef safe_import(name,globals=None,locals=None,fromlist=(),level=0):\n    if level or name not in ALLOWED_IMPORTS:raise ImportError(\'Blocked import\')\n    if name not in _MODULES:\n        _MODULES[name]=SimpleNamespace(**{n:getattr(math,n)for n in dir(math)if not n.startswith(\'_\')})if name==\'math\'else safe_sympy()\n    return _MODULES[name]\n\ndef worker_init(memory_bytes):\n    # Spawned workers import no Torch. No GPU state or model is inherited.\n    resource.setrlimit(resource.RLIMIT_CORE,(0,0))\n    if sys.platform.startswith(\'linux\') and hasattr(resource,\'RLIMIT_AS\'):\n        soft,hard=resource.getrlimit(resource.RLIMIT_AS)\n        value=memory_bytes if hard==resource.RLIM_INFINITY else min(memory_bytes,hard)\n        resource.setrlimit(resource.RLIMIT_AS,(value,value))\n    signal.signal(signal.SIGALRM,lambda signum,frame:(_ for _ in ()).throw(ProgramTimeout()))\n\ndef execute_task(payload):\n    code,wall_limit,cpu_limit=payload\n    started=time.monotonic()\n    tree,status=filter_program(code)\n    if status:return {\'status\':status}\n    # RLIMIT_CPU is cumulative per process. Set a new SOFT deadline for EACH task;\n    # retain the hard limit so the soft deadline can increase for reused workers.\n    used=resource.getrusage(resource.RUSAGE_SELF)\n    soft=max(1,math.ceil(used.ru_utime+used.ru_stime)+int(math.ceil(cpu_limit)))\n    _,hard=resource.getrlimit(resource.RLIMIT_CPU)\n    if hard!=resource.RLIM_INFINITY:soft=min(soft,hard)\n    resource.setrlimit(resource.RLIMIT_CPU,(soft,hard))\n    signal.setitimer(signal.ITIMER_REAL,wall_limit)\n    try:\n        env={\'__builtins__\':dict(SAFE_BUILTINS,__import__=safe_import)}\n        exec(compile(tree,\'<bounded-dataset-program>\',\'exec\'),env,env)\n        solve=env.get(\'solve\')\n        if not callable(solve):return {\'status\':\'missing_solve\'}\n        value=solve()\n        if isinstance(value,bool):value=int(value)\n        if not isinstance(value,(int,float)):\n            # SymPy real scalars may be used; containers/strings/complex refused.\n            if type(value).__module__.startswith(\'sympy\') and value.is_real is True:value=float(value)\n            else:return {\'status\':\'non_number\'}\n        value=float(value)\n        if not math.isfinite(value):return {\'status\':\'non_finite\'}\n        return {\'status\':\'numeric\',\'value\':value}\n    except ProgramTimeout:return {\'status\':\'timeout\'}\n    except BaseException as e:return {\'status\':\'execution_error\',\'error_type\':type(e).__name__}\n    finally:signal.setitimer(signal.ITIMER_REAL,0)\n\ndef create_pool(workers,memory_bytes):\n    return mp.get_context(\'spawn\').Pool(workers,initializer=worker_init,\n        initargs=(memory_bytes,),maxtasksperchild=256)\n\ndef evaluate_programs(codes,workers=2,wall_limit=1.0,cpu_limit=2.0,\n                      parent_timeout=3.0,memory_bytes=768*1024**2,deadline=None,progress=None):\n    """Bounded inflight tasks; every async result has a parent deadline.\n    A killed worker loses its task even if Pool replaces it: terminate/recreate\n    the WHOLE pool, retain completed results and retry collateral pending tasks.\n    Pool.map is deliberately never used. Output order and duplicates preserved.\n    """\n    if parent_timeout<=wall_limit:raise ValueError(\'Parent deadline must exceed program deadline\')\n    unique=list(dict.fromkeys(codes));todo=deque(range(len(unique)));answers={};pending={}\n    pool=None;restarts=0;start=time.monotonic();cancelled=False;last_progress=-1\n    try:\n        while todo or pending:\n            if deadline is not None and time.monotonic()>=deadline:\n                cancelled=True;break\n            if pool is None:pool=create_pool(workers,memory_bytes)\n            while todo and len(pending)<workers:\n                i=todo.popleft();tree,status=filter_program(unique[i])\n                if status:answers[i]={\'status\':status};continue\n                pending[i]=(pool.apply_async(execute_task,((unique[i],wall_limit,cpu_limit),)),time.monotonic())\n            for i,(future,submitted)in list(pending.items()):\n                if future.ready():\n                    try:answers[i]=future.get(timeout=0)\n                    except Exception as e:answers[i]={\'status\':\'worker_error\',\'error_type\':type(e).__name__}\n                    del pending[i]\n            timed=[i for i,(_,submitted)in pending.items()if time.monotonic()-submitted>=parent_timeout]\n            if timed:\n                for i in timed:answers[i]={\'status\':\'timeout_or_worker_exit\'}\n                retry=[i for i in pending if i not in timed]\n                pool.terminate();pool.join();pool=None;restarts+=1;pending.clear()\n                todo.extendleft(reversed(retry))\n            if progress and len(answers)!=last_progress and (len(answers)%1000==0 or len(answers)==len(unique)):\n                progress(len(answers),len(unique));last_progress=len(answers)\n            if pending:time.sleep(.001)\n        if cancelled:\n            for i in todo:answers[i]={\'status\':\'budget_not_evaluated\'}\n            for i in pending:answers[i]={\'status\':\'budget_not_evaluated\'}\n    finally:\n        if pool is not None:pool.terminate();pool.join()\n    by_code={c:answers.get(i,{\'status\':\'budget_not_evaluated\'})for i,c in enumerate(unique)}\n    rows=[dict(by_code[c])for c in codes]\n    return {\'rows\':rows,\'summary\':{\'programs\':len(codes),\'unique_programs\':len(unique),\n        \'statuses\':dict(Counter(r[\'status\']for r in rows)),\'pool_restarts\':restarts,\n        \'seconds\':time.monotonic()-start,\'complete\':not cancelled}}\n\ndef compare_exec(reference_results,generation_results,rel_tol=1e-6,abs_tol=1e-6):\n    refs=reference_results[\'rows\']if isinstance(reference_results,dict)else reference_results\n    gens=generation_results[\'rows\']if isinstance(generation_results,dict)else generation_results\n    if len(refs)!=len(gens):raise ValueError(\'Order/length mismatch\')\n    eligible=[i for i,r in enumerate(refs)if r[\'status\']==\'numeric\']\n    matches=[i for i in eligible if gens[i][\'status\']==\'numeric\'and math.isclose(\n        gens[i][\'value\'],refs[i][\'value\'],rel_tol=rel_tol,abs_tol=abs_tol)]\n    return {\'denominator\':len(eligible),\'matches\':len(matches),\n        \'accuracy\':len(matches)/len(eligible)if eligible else None,\n        \'eligible_indices\':eligible,\'match_indices\':matches,\n        \'reference_statuses\':dict(Counter(r[\'status\']for r in refs)),\n        \'generation_statuses\':dict(Counter(r[\'status\']for r in gens)),\n        \'rel_tol\':rel_tol,\'abs_tol\':abs_tol}\n', 'bench23_data.py': 'import hashlib,json,random,time\nfrom pathlib import Path\nfrom array import array\nfrom urllib.request import urlretrieve\nimport pyarrow.parquet as pq\nfrom bench23_core import sha_file,write_json,encode,BOS,EOS,check_deadline\nREVISION=\'b2724689ff079e9c22efd548fe134bfb29036dbb\'\nSHARDS=[\n {\'path\':\'data/train/part-00000.parquet\',\'bytes\':26169796,\'sha256\':\'91831d582faa12d73b1a6b3878e496446cd5df074f17ee4ffea8cc2887c708ec\'},\n {\'path\':\'data/train/part-00001.parquet\',\'bytes\':26166175,\'sha256\':\'5222d657a1af07a531f1b22503f35741a6547f0831509897d0a279bc8f67d068\'},\n {\'path\':\'data/train/part-00002.parquet\',\'bytes\':26162244,\'sha256\':\'c6082401ea966aeaa8375018ae1f1d9c7c55cc30e421c45cc933ceacd97dc3de\'}]\nTRAIN_HASH=\'61483f4dc6970ff5f2b1afffa45d57391dcfc01ab6ec4f140c6e6b6c792c8c38\'\nLOCATIONS_HASH=\'c2332a54a923ffa85da9ac91ec0a39a2e4ead7f7c2e328abecef702e1d306ba7\'\nHELDOUT_HASH=\'e187bc297218365f1a1452a15376e0ff2fceaf8a5a5517820549854b90564743\'\nTEST_HASH=\'8bde59c8d8bb7edc4e8a503be22fc132c4a47773025d333432e78c5489e6522a\'\n\ndef data_hash(rows):return hashlib.sha256(json.dumps(rows,ensure_ascii=False,sort_keys=True).encode()).hexdigest()\ndef is_validation(question):return int.from_bytes(hashlib.sha256(f\'42:{question}\'.encode()).digest()[:8],\'big\')%10==0\ndef eligible(row):return len(row[\'question\'].encode())<=512 and len(row[\'code\'].encode())<512\n\ndef download_checked(spec,cache,deadline=None):\n check_deadline(deadline);cache=Path(cache);cache.mkdir(parents=True,exist_ok=True);target=cache/Path(spec[\'path\']).name\n if not target.exists():\n  tmp=target.with_suffix(\'.download\')\n  # Network calls have a bounded socket timeout; deadline checked again afterward.\n  import socket\n  socket.setdefaulttimeout(30)\n  urlretrieve(f\'https://huggingface.co/datasets/SlavaYus/tinysms_qa_12m/resolve/{REVISION}/{spec["path"]}\',tmp)\n  if sha_file(tmp)!=spec[\'sha256\']:raise ValueError(\'Downloaded SHA mismatch\')\n  tmp.replace(target)\n if sha_file(target)!=spec[\'sha256\']:raise ValueError(\'Cached SHA mismatch\')\n if \'bytes\'in spec and target.stat().st_size!=spec[\'bytes\']:raise ValueError(\'Size mismatch\')\n check_deadline(deadline);return target\n\ndef prepare_data(cache,tokenizer,deadline=None,progress=print):\n started=time.monotonic()\n test_file=download_checked({\'path\':\'data/test/test.parquet\',\'sha256\':TEST_HASH},cache,deadline)\n test=[{\'question\':q}for q in pq.read_table(test_file,columns=[\'question\'])[\'question\'].to_pylist()]\n assert len(test)==1919;test_questions={r[\'question\']for r in test}\n train=[];locations=[];validation=None;heldout_pool=[];heldout_questions=set();paths=[]\n # Exact21/22 selection. No reference-exec filtering in this throughput experiment.\n for shard_index,spec in enumerate(SHARDS):\n  path=download_checked(spec,cache,deadline);paths.append(path);offset=0\n  for batch in pq.ParquetFile(path).iter_batches(batch_size=4096):\n   check_deadline(deadline);rows=batch.to_pylist()\n   if validation is None:\n    validation=[r for r in rows if eligible(r)and is_validation(r[\'question\'])][:64]\n    assert len(validation)==64;heldout_questions.update(r[\'question\']for r in validation)\n   for i,row in enumerate(rows):\n    if not eligible(row)or row[\'question\']in test_questions:continue\n    if is_validation(row[\'question\']):\n     if shard_index==0 and row[\'question\']not in heldout_questions and len(heldout_pool)<960:\n      heldout_pool.append(row);heldout_questions.add(row[\'question\'])\n     continue\n    train.append(row);locations.append((shard_index,offset+i))\n    if len(train)==500000:break\n   if len(train)==500000:break\n   offset+=len(rows)\n  if progress:progress(f\'Data shard{shard_index}: selected {len(train)} /500000\')\n  if len(train)==500000:break\n heldout=validation+heldout_pool\n assert len(train)==500000 and len(heldout)==1024 and len(heldout_questions)==1024\n assert data_hash(train)==TRAIN_HASH and data_hash(locations)==LOCATIONS_HASH and data_hash(heldout)==HELDOUT_HASH\n assert not any(r[\'question\']in heldout_questions or r[\'question\']in test_questions for r in train)\n summary={\'revision\':REVISION,\'train_rows\':len(train),\'heldout_rows\':len(heldout),\'test_rows\':len(test),\n  \'train_sha256\':TRAIN_HASH,\'locations_sha256\':LOCATIONS_HASH,\'heldout_sha256\':HELDOUT_HASH,\'shards\':SHARDS,\n  \'selection_seconds\':time.monotonic()-started,\'filter_reference_non_number\':False,\'fresh_heldout_not_evaluated\':True}\n # encode_batch keeps the unchanged tokenization but avoids a Python call per string.\n token_cache=[];encoding_started=time.monotonic()\n for lo in range(0,len(train),2048):\n  check_deadline(deadline);part=train[lo:lo+2048]\n  questions=tokenizer.encode_batch([r[\'question\']for r in part],add_special_tokens=False)\n  codes=tokenizer.encode_batch([r[\'code\']for r in part],add_special_tokens=False)\n  token_cache.extend((array(\'H\',[BOS]+[i+3 for i in q.ids]+[EOS]),array(\'H\',[i+3 for i in c.ids]))for q,c in zip(questions,codes))\n summary[\'bpe_cache_seconds\']=time.monotonic()-encoding_started\n summary[\'source_tokens\']=sum(len(x[0])for x in token_cache);summary[\'target_tokens\']=sum(len(x[1])+1 for x in token_cache)\n return token_cache,heldout,test,paths,summary\n\ndef audit_train50k(paths,evaluate_programs,deadline=None):\n codes=[];locations=[]\n for shard,path in enumerate(paths):\n  check_deadline(deadline);table=pq.read_table(path,columns=[\'code\']);n=16667 if shard<2 else 16666\n  indices=sorted(random.Random(230042+shard).sample(range(len(table)),n));column=table[\'code\']\n  for i in indices:codes.append(column[i].as_py());locations.append((shard,i))\n assert len(codes)==50000\n run=evaluate_programs(codes,deadline=deadline)\n numeric=run[\'summary\'][\'statuses\'].get(\'numeric\',0);blocked=sum(v for k,v in run[\'summary\'][\'statuses\'].items()if k.startswith(\'blocked_\'))\n return {\'revision\':REVISION,\'selection\':\'uniform raw rows16667/16667/16666; seed230042+shard, no eligibility filter\',\n  \'sample_rows\':len(codes),\'locations_sha256\':data_hash(locations),\'finite\':numeric,\'not_finite_or_failed\':len(codes)-numeric,\n  \'fraction_not_finite_or_failed\':1-numeric/len(codes),\'AST_blocked_unknown\':blocked,\'execution\':run[\'summary\']}\n', 'bench23_input.py': "import hashlib,json,shutil,zipfile\nfrom pathlib import Path\nfrom bench23_core import sha_file,write_json\nZIP_SHA='4db40c0313f10afbf11d4a76beb2de59a729e1af9c6d40f08240c6f55697c615'\nPT_SHAS={'5d115cf06c55f8d6245b5244d4cd622e38480fd5ef7dfe8806ba8bbcd70d4833':'step-100000',\n '13f8ae4ae2ee2c15fc9d034cf8773de420226e073550c9774adb1846f3e27b23':'latest-100000'}\nTOKENIZER_SHA='261f25b5d4be006a6a00e41e0eafbe522580741da627ba670f6b9805a2659b7d'\n\ndef verify_inputs(paths,destination):\n destination=Path(destination);destination.mkdir(parents=True,exist_ok=True);paths=[Path(p)for p in paths]\n manifest={'source':'user-upload','files':[]};pt=None;tokenizer=None\n if len(paths)==1 and paths[0].suffix.lower()=='.zip':\n  digest=sha_file(paths[0])\n  if digest!=ZIP_SHA:raise ValueError('ZIP is not the original22 archive: SHA mismatch, preparation stopped')\n  manifest['zip_sha256']=digest\n  # Never extractall; select two files by their exact content hashes, prevent zip slip.\n  with zipfile.ZipFile(paths[0])as archive:\n   for info in archive.infolist():\n    if info.is_dir()or info.file_size>150_000_000:continue\n    if not (info.filename.endswith('.pt')and ('100000'in info.filename or Path(info.filename).name=='latest.pt')or info.filename.endswith('.json')and'tokenizer'in info.filename.lower()):continue\n    with archive.open(info)as f:\n     h=hashlib.sha256()\n     for chunk in iter(lambda:f.read(1024**2),b''):h.update(chunk)\n    file_sha=h.hexdigest()\n    kind='checkpoint'if file_sha in PT_SHAS else'tokenizer'if file_sha==TOKENIZER_SHA else None\n    if kind is None:continue\n    if kind=='checkpoint'and pt is not None:continue\n    if kind=='tokenizer'and tokenizer is not None:continue\n    target=destination/('source22-step100000.pt'if kind=='checkpoint'else'source22-tokenizer.json')\n    with archive.open(info)as source,target.open('wb')as out:shutil.copyfileobj(source,out)\n    if sha_file(target)!=file_sha:raise ValueError('Extracted member SHA mismatch')\n    manifest['files'].append({'kind':kind,'zip_member':info.filename,'sha256':file_sha,'bytes':target.stat().st_size})\n    if kind=='checkpoint':pt=target\n    else:tokenizer=target\n else:\n  for path in paths:\n   file_sha=sha_file(path)\n   if file_sha in PT_SHAS:\n    if pt is not None:raise ValueError('Supply only one checkpoint')\n    pt=path;kind='checkpoint'\n   elif file_sha==TOKENIZER_SHA:\n    if tokenizer is not None:raise ValueError('Supply only one tokenizer')\n    tokenizer=path;kind='tokenizer'\n   else:raise ValueError(f'Unexpected upload / SHA mismatch: {path.name}')\n   manifest['files'].append({'kind':kind,'filename':path.name,'sha256':file_sha,'bytes':path.stat().st_size})\n if pt is None or tokenizer is None:raise ValueError('Need22/100k PT and tokenizer JSON, or original ZIP22')\n manifest['verified_before_data_and_weights']=True\n return pt,tokenizer,manifest\n"}
CPU_SAVED_EXEC_CHECK={'date': '2026-10-09', 'scope': 'New CPU execution of saved user generations; zero model training updates', 'reference': {'programs': 1024, 'unique_programs': 1023, 'statuses': {'numeric': 991, 'non_number': 33}, 'pool_restarts': 0, 'seconds': 0.8926786249503493, 'complete': True}, 'scores': {'18/100000': {'matches': 151, 'denominator': 991, 'accuracy': 0.15237134207870837}, '19/100000': {'matches': 145, 'denominator': 991, 'accuracy': 0.14631685166498487}, '20/100000': {'matches': 184, 'denominator': 991, 'accuracy': 0.1856710393541877}, '21/50000': {'matches': 156, 'denominator': 991, 'accuracy': 0.1574167507568113}, '21/75000': {'matches': 176, 'denominator': 991, 'accuracy': 0.17759838546922302}, '21/100000': {'matches': 193, 'denominator': 991, 'accuracy': 0.19475277497477295}, '22/100000': {'matches': 189, 'denominator': 991, 'accuracy': 0.19071644803229063}}, 'source_JSON_sha256': {'18': '4f41c616a7bcdf73d6f6620c3933c5860ace831df36a1d33e89a73804f119708', '19': '164f208fd91ac4f69fb46a6443295ad8e83243909aa3fb3315682f29dd430084', '20': '99a37dac876e4c06b372e80f5084c1580f1298f86e18f3e740cc6070837d8e2d', '21': '6149e01ffde91b3aea844abf2e55437df8ece1b973fa5a4148b45fdf6a204056', '22': '3a4e687055e579715dbf45ff8bea22b01decae37e7ffc4f460aa61385e950a33'}}
CPU_TRAIN50K_AUDIT={'date': '2026-10-09', 'pinned_revision': 'b2724689ff079e9c22efd548fe134bfb29036dbb', 'selection': 'Uniform row sample of raw shards0..2:16667/16667/16666, per-shard seed230042+shard. Includes long/heldout-like raw train rows; no training rows filtered in notebook23.', 'sample_rows': 50000, 'locations_sha256': 'e817d6d4aeca88e6dd04f964bb0edea81640beaf730ecf320f814f740d82ffd4', 'source_shard_sha256': ['91831d582faa12d73b1a6b3878e496446cd5df074f17ee4ffea8cc2887c708ec', '5222d657a1af07a531f1b22503f35741a6547f0831509897d0a279bc8f67d068', 'c6082401ea966aeaa8375018ae1f1d9c7c55cc30e421c45cc933ceacd97dc3de'], 'finite': 48210, 'not_finite_or_failed': 1790, 'fraction_not_finite_or_failed': 0.035800000000000054, 'AST_blocked_unknown': 60, 'execution': {'programs': 50000, 'unique_programs': 49561, 'statuses': {'numeric': 48210, 'non_number': 1727, 'blocked_import': 54, 'execution_error': 3, 'blocked_name': 6}, 'pool_restarts': 0, 'seconds': 36.58295991597697, 'complete': True}, 'limitations': 'Restricted numeric sandbox permits math and safe numeric SymPy subset. AST blocks reported separately; blocked/timeout not proof of semantically invalid reference. No new training labels; audit is for future filtering.', 'new_training_updates': 0}

for filename,source in MODULE_SOURCES.items():
    (MODULE_DIR/filename).write_text(source,encoding='utf-8')
sys.path.insert(0,str(MODULE_DIR.resolve()))
import bench23_core as bench
import bench23_data as data
import bench23_input as inputs
import bench23_exec as numeric
import torch,tokenizers,pyarrow,sympy
import pandas as pd
assert tokenizers.__version__=='0.23.2'
assert torch.cuda.is_available(),'Выбери T4 в Runtime → Change runtime type'
assert torch.cuda.device_count()==1 and 'T4' in torch.cuda.get_device_name(0),'Протокол рассчитан на одну T4'
device=torch.device('cuda')
torch.set_num_threads(2)
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False
torch.set_float32_matmul_precision('highest')
RUN_DIR=Path('hw02-bench23-results')
RUN_DIR.mkdir(exist_ok=True)
WORK_DEADLINE=_BENCH23_STARTED+59*60
TRAIN_DEADLINE=WORK_DEADLINE-20*60
result={'experiment':23,'status':'setup','runtime':{
    'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__,'cuda':torch.version.cuda,
    'tokenizers':tokenizers.__version__,'pyarrow':pyarrow.__version__,'sympy':sympy.__version__,'cpu_threads':torch.get_num_threads(),
    'SDPA_enabled_flags':bench.sdpa_flags(),'TF32':False},
    'protocol':{'total_budget_minutes':60,'work_minutes':59,'inference_reserve_minutes':20,
        'train_warmup':50,'train_measured':200,'bucket_chunk_pairs':4096,'train_LR':1e-4,
        'train_seed':230042,'numeric_workers':2,'program_wall_seconds':1.,'program_CPU_seconds':2.,
        'parent_task_seconds':3.,'train_memory_limit_bytes':14_000_000_000,
        'train_selection':'useful target tokens/s, finite loss/grads,200 steps, no measured AMP skips',
        'inference_selection':'minimum test full wall; exec-match drop<=0.005 versus control, denom991',
        'cap':512,'done_CPU_check_every_optimized':16,'quality_training':False},
    'CPU_saved_generation_check':CPU_SAVED_EXEC_CHECK,'CPU_train50k_audit':CPU_TRAIN50K_AUDIT,
    'train':[],'inference':[],'attempts':[]}
def save():
    result['elapsed_seconds_since_first_cell']=time.monotonic()-_BENCH23_STARTED
    bench.write_json(RUN_DIR/'results.json',result)
save()
bench.write_json(RUN_DIR/'cpu-saved-exec-recheck.json',CPU_SAVED_EXEC_CHECK)
bench.write_json(RUN_DIR/'cpu-train50k-audit.json',CPU_TRAIN50K_AUDIT)
print('RUNTIME',json.dumps(result['runtime'],ensure_ascii=False))
print(subprocess.run(['nvidia-smi','--query-gpu=name,driver_version,memory.total,memory.free,temperature.gpu,clocks.sm','--format=csv'],capture_output=True,text=True).stdout)
print('Saved CPU exec-match:',CPU_SAVED_EXEC_CHECK['scores'])
print('CPU audit50k:',CPU_TRAIN50K_AUDIT['execution']['statuses'])
PARAMETERS={}
for name,spec in bench.MODEL_SPECS.items():
    with torch.device('meta'):
        model_meta=bench.SimpleTransformer(**spec)
    PARAMETERS[name]=sum(p.numel() for p in model_meta.parameters())
    del model_meta
result['model_parameters']=PARAMETERS
print('PARAMETERS',PARAMETERS)
save()

RUNTIME {"gpu": "Tesla T4", "torch": "2.11.0+cu130", "cuda": "13.0", "tokenizers": "0.23.2", "pyarrow": "23.0.1", "sympy": "1.14.0", "cpu_threads": 2, "SDPA_enabled_flags": {"flash": true, "mem_efficient": true, "math": true, "cudnn": true}, "TF32": false}
name, driver_version, memory.total [MiB], memory.free [MiB], temperature.gpu, clocks.current.sm [MHz]
Tesla T4, 580.82.07, 15360 MiB, 14910 MiB, 41, 300 MHz

Saved CPU exec-match: {'18/100000': {'matches': 151, 'denominator': 991, 'accuracy': 0.15237134207870837}, '19/100000': {'matches': 145, 'denominator': 991, 'accuracy': 0.14631685166498487}, '20/100000': {'matches': 184, 'denominator': 991, 'accuracy': 0.1856710393541877}, '21/50000': {'matches': 156, 'denominator': 991, 'accuracy': 0.1574167507568113}, '21/75000': {'matches': 176, 'denominator': 991, 'accuracy': 0.17759838546922302}, '21/100000': {'matches': 193, 'denominator': 991, 'accuracy': 0.19475277497477295}, '22/100000': {'matches': 189, 'denominator': 991, 'accuracy': 

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(


## 2. Загрузить и сверить22/100k

Выбрать исходный ZIP22 **или** два файла PT+JSON. SHA сверяются с сохранённой сводкой22, не с содержимым загруженного ZIP-manifest. Архив распаковывается выборочно, веса лежат вне каталога результатов. В ZIP23 веса не попадут.

In [3]:
from google.colab import files
try:
    uploaded=files.upload()
    UPLOAD_DIR=Path('hw02-bench23-input')
    UPLOAD_DIR.mkdir(exist_ok=True)
    uploaded_paths=[]
    for filename,content in uploaded.items():
        target=UPLOAD_DIR/Path(filename).name
        target.write_bytes(content)
        uploaded_paths.append(target)
    del uploaded
    CHECKPOINT,TOKENIZER_PATH,manifest=inputs.verify_inputs(uploaded_paths,UPLOAD_DIR/'verified')
    result['input']=manifest
    print('INPUT VERIFIED',json.dumps(manifest,ensure_ascii=False,indent=2))
    result['status']='input_verified'
    save()
except BaseException as error:
    result['status']='input_failed_or_interrupted'
    result['input_error']=type(error).__name__+': '+str(error)[:400]
    save()
    raise

Saving hw02-encoder4-decoder4-qwen-bpe8k-swiglu-500k-cosine-tail-75000-to-100000-results.zip to hw02-encoder4-decoder4-qwen-bpe8k-swiglu-500k-cosine-tail-75000-to-100000-results.zip
INPUT VERIFIED {
  "source": "user-upload",
  "files": [
    {
      "kind": "checkpoint",
      "zip_member": "latest.pt",
      "sha256": "13f8ae4ae2ee2c15fc9d034cf8773de420226e073550c9774adb1846f3e27b23",
      "bytes": 146269451
    },
    {
      "kind": "tokenizer",
      "zip_member": "tokenizer.json",
      "sha256": "261f25b5d4be006a6a00e41e0eafbe522580741da627ba670f6b9805a2659b7d",
      "bytes": 545866
    }
  ],
  "zip_sha256": "4db40c0313f10afbf11d4a76beb2de59a729e1af9c6d40f08240c6f55697c615",
  "verified_before_data_and_weights": true
}


## 3. Pinned данные / кеш и reference-исполнение

Те же500k train, heldout1024, test1919, revision `b2724689ff079e9c22efd548fe134bfb29036dbb`, прежниеSHA и отбор. Reference-фильтр **не меняет** train23. Fresh1024 не оценивается.

Аудит50k уже выполнен на CPU при подготовке: **48210 конечных;1790 не дали число (3.58%)**, включая60 AST-заблокированных с неизвестным результатом. Чтобы повторить именно этот CPU-аудит в Colab, поставить `REPEAT_TRAIN50K_AUDIT=True` до запуска ячейки; он входит в60 минут. Проверка991 reference выполняется заново в любом случае. Программы исполняются на CPU, не на T4.

In [4]:
REPEAT_TRAIN50K_AUDIT=False
try:
    tokenizer,tokenbytes=bench.init_tokenizer(TOKENIZER_PATH)
    token_cache,heldout,test,shard_paths,data_summary=data.prepare_data('hw02-data',tokenizer,WORK_DEADLINE)
    result['data']=data_summary
    print('DATA',json.dumps(data_summary,indent=2))
    # Quick safety/protocol fixtures before the real programs.
    fixture=numeric.evaluate_programs(['def solve():\n return True','import math\ndef solve():\n return math.sqrt(4)',
        'import os\ndef solve():\n return 1','def solve():\n return float("inf")'],deadline=WORK_DEADLINE)
    assert [r['status'] for r in fixture['rows']]==['numeric','numeric','blocked_import','non_finite']
    assert fixture['rows'][0]['value']==1
    result['sandbox_fixture']=fixture['summary']
    reference_exec=numeric.evaluate_programs([r['code'] for r in heldout],deadline=WORK_DEADLINE)
    assert reference_exec['summary']['complete'] and reference_exec['summary']['statuses'].get('numeric')==991,reference_exec['summary']
    bench.write_json(RUN_DIR/'heldout-reference-execution.json',reference_exec)
    result['reference_exec']=reference_exec['summary']
    print('REFERENCE',json.dumps(reference_exec['summary']))
    if REPEAT_TRAIN50K_AUDIT:
        audit=data.audit_train50k(shard_paths,numeric.evaluate_programs,WORK_DEADLINE)
        result['train50k_repeated_in_Colab']=audit
        bench.write_json(RUN_DIR/'train50k-repeated-in-colab.json',audit)
        print('TRAIN50K COLAB CPU',json.dumps(audit,indent=2))
    # Actual observed SDPA operators for encoder/cached decoder, separately from timers.
    sdpa_profiles={}
    for precision in ['fp32','fp16']:
        bench.check_deadline(WORK_DEADLINE)
        state=torch.load(CHECKPOINT,map_location='cpu',weights_only=False)
        assert state['steps']==100000 and state['tokenizer_sha256']==inputs.TOKENIZER_SHA
        model=bench.SimpleTransformer(**bench.MODEL_SPECS['A'])
        model.load_state_dict(state['model'],strict=True)
        del state
        model=model.to(device).eval()
        if precision=='fp16':model.half()
        src=bench.source_batch(tokenizer,[heldout[0]['question']],device)
        sdpa_profiles[precision]=bench.profile_sdpa(lambda:bench.generate_preallocated(model,src,8),device)
        del model,src
        gc.collect();torch.cuda.empty_cache()
    result['inference_SDPA_preflight_outside_timers']=sdpa_profiles
    print('OBSERVED INFERENCE SDPA',json.dumps(sdpa_profiles))
    result['status']='data_reference_ready'
    save()
except BaseException as error:
    result['status']='setup_failed_or_interrupted'
    result['setup_error']=type(error).__name__+': '+str(error)[:400]
    save()
    raise

Data shard0: selected 174519 /500000
Data shard1: selected 349364 /500000
Data shard2: selected 500000 /500000
DATA {
  "revision": "b2724689ff079e9c22efd548fe134bfb29036dbb",
  "train_rows": 500000,
  "heldout_rows": 1024,
  "test_rows": 1919,
  "train_sha256": "61483f4dc6970ff5f2b1afffa45d57391dcfc01ab6ec4f140c6e6b6c792c8c38",
  "locations_sha256": "c2332a54a923ffa85da9ac91ec0a39a2e4ead7f7c2e328abecef702e1d306ba7",
  "heldout_sha256": "e187bc297218365f1a1452a15376e0ff2fceaf8a5a5517820549854b90564743",
  "shards": [
    {
      "path": "data/train/part-00000.parquet",
      "bytes": 26169796,
      "sha256": "91831d582faa12d73b1a6b3878e496446cd5df074f17ee4ffea8cc2887c708ec"
    },
    {
      "path": "data/train/part-00001.parquet",
      "bytes": 26166175,
      "sha256": "5222d657a1af07a531f1b22503f35741a6547f0831509897d0a279bc8f67d068"
    },
    {
      "path": "data/train/part-00002.parquet",
      "bytes": 26162244,
      "sha256": "c6082401ea966aeaa8375018ae1f1d9c7c55cc30e421c4

/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


OBSERVED INFERENCE SDPA {"fp32": {"status": "observed", "operators": ["aten::_scaled_dot_product_efficient_attention", "aten::scaled_dot_product_attention"], "scope": "one warmup forward/backward for this shape/dtype, outside measured steps; backend may change with shapes"}, "fp16": {"status": "observed", "operators": ["aten::_scaled_dot_product_efficient_attention", "aten::scaled_dot_product_attention"], "scope": "one warmup forward/backward for this shape/dtype, outside measured steps; backend may change with shapes"}}


## 4. Greedy-контроль22: исходный порядок / batch32 / FP32

Контроль проверяем первым. Каждый split отдельно загружает те же веса. Старый таймер: encode/H2D/encoder/KV/CPU-decode/AST; новый full wall: `torch.load`→restore/cast/H2D→warmup→generation→файл ответов. Exec-оценка идёт отдельно после таймера. Время загрузки датасета/кеша исключено из обоих. SDPA-профилирование вынесено за таймеры.

Контроль использует `torch.cat` и проверку done на каждом шаге, как21/22. Варианты сортируют по длине вопроса, используют предаллоцированный выход, проверяют done каждые16 шагов, возвращают исходный порядок. Число шагов: сумма итераций decoder по батчам; дополнительно batch×iterations. PAD послеEOS не участвует в token-identical.

In [5]:
INFERENCE_VARIANTS=[{'id':'control-fp32-b32','path':'control','batch_pairs':32,'precision':'fp32'}]+[
    {'id':f'sorted-{precision}-b{batch}','path':'preallocated','batch_pairs':batch,'precision':precision}
    for batch in [64,128,256,512] for precision in ['fp32','fp16']]
BASELINE={}
def infer_variant(variant,require_control=False):
    row={'id':variant['id'],'config':variant,'status':'started'}
    result['attempts'].append({'kind':'inference','id':variant['id'],'status':'started'})
    save()
    try:
        for split,rows in [('heldout',heldout),('test',test)]:
            summary,examples=bench.run_inference_split(CHECKPOINT,TOKENIZER_PATH,rows,split,variant,device,
                RUN_DIR/'answers',deadline=WORK_DEADLINE,baseline=BASELINE.get(split))
            row[split]=summary
            if summary['status']!='ok':
                row['status']=summary['status']
                return row
            # Save each completed split immediately; interruption leaves useful outputs.
            bench.write_json(RUN_DIR/'generations'/f'{variant["id"]}-{split}.json',examples)
            if split=='heldout':
                executed=numeric.evaluate_programs([e['generated_code'] for e in examples],deadline=WORK_DEADLINE)
                score=numeric.compare_exec(reference_exec,executed)
                summary['exec_match']=score
                summary['exec_seconds_outside_timers']=executed['summary']['seconds']
                summary['exec_complete']=executed['summary']['complete']
                bench.write_json(RUN_DIR/'execution'/f'{variant["id"]}-heldout.json',executed)
                if not executed['summary']['complete']:
                    row['status']='budget_exec'
                    return row
                if require_control:
                    assert score['denominator']==991 and abs(score['matches']-189)<=3,score
                    result['control_quality_gate_passed']=True
                print('EXEC',variant['id'],score['matches'],score['denominator'],flush=True)
            if require_control:BASELINE[split]=examples
            save()
        row['status']='ok'
        return row
    except bench.BudgetExpired:
        row['status']='budget'
        return row
    except BaseException as error:
        row['status']='interrupted' if isinstance(error,KeyboardInterrupt) else 'error'
        row['error']=type(error).__name__+': '+str(error)[:400]
        raise
    finally:
        result['inference'].append(row)
        result['attempts'][-1]['status']=row['status']
        bench.write_json(RUN_DIR/'inference'/f'{variant["id"]}.json',row)
        save()

if result['inference']:raise RuntimeError('Контроль уже запускался; используйте свежую сессию')
try:
    control=infer_variant(INFERENCE_VARIANTS[0],require_control=True)
    assert control['status']=='ok' and result.get('control_quality_gate_passed'),control
    result['status']='inference_control_ready'
    save()
    print('CONTROL',control['id'],'heldout',control['heldout']['exec_match']['matches'],'/991',
        'test old/full seconds',control['test']['seconds_historical_scope'],control['test']['seconds_full_wall'])
except BaseException as error:
    result['status']='control_failed_or_interrupted'
    result['control_error']=type(error).__name__+': '+str(error)[:400]
    save()
    raise

EXEC control-fp32-b32 189 991
CONTROL control-fp32-b32 heldout 189 /991 test old/full seconds 70.87654636200023 71.21665250700016


## 5. Сетка обучения:50+200 шагов на конфигурацию

Каждый запуск создаёт новые случайные веса и новый optimizer; качество/LR-план будущего обучения отсюда не выбираем. Таймер CUDA-sync по чанкам20 шагов включает collate/H2D/loss/backward/unscale/clip/AdamW/GradScaler. Планы FP32/AMP общие. Полезные source-токены включают BOS/EOS, target — EOS; PAD исключён. Padding считается по реально измеренным slots. AMP-skips считаются по one-device found_inf GradScaler, сохраняются на GPU до clip и читаются после таймера; диагностический API проверяется перед запуском. Peak allocated и reserved включают веса/градиенты/optimizer и весь прогрев. SDPA-профиль первого прогревочного шага сохраняется отдельно.

OOM не останавливает сетку. Если проекция после50 прогревочных шагов не помещается в дедлайн обучения, конфигурация явно помечается `budget_projection`. Допустимы только200 полностью измеренных шагов. Размер памяти — decimalGB для правила14GB; дополнительно показывается GiB.

In [6]:
if result['train']:raise RuntimeError('Сетка уже запускалась; используйте свежую сессию')
result['status']='training_benchmark_running'
save()
try:
    fused=bench.fused_adamw_available(device)
    print('FUSED ADAMW AVAILABLE',fused)
    random8=bench.random_plan(len(token_cache),8,250)
    bucket_plans={batch:bench.bucket_plan(token_cache,batch,250,chunk_pairs=4096) for batch in [32,128,256]}
    plan_hashes={str(batch):data.data_hash(plan) for batch,plan in bucket_plans.items()}
    result['batch_plans']={'random8_sha256':data.data_hash(random8),'bucketed_sha256':plan_hashes,
        'shared_across_models_and_FP32_AMP':True}
    TRAIN_CONFIGS=[{'id':'A-fp32-b8-random-control','model':'A','precision':'fp32','batch_pairs':8,'bucketed':False,'fused':False},
        {'id':'A-amp_fp16-b8-random','model':'A','precision':'amp_fp16','batch_pairs':8,'bucketed':False,'fused':False}]+[
        {'id':f'{name}-{precision}-b{batch}-bucketed','model':name,'precision':precision,'batch_pairs':batch,'bucketed':True,'fused':fused}
        for name in ['A','B','C'] for batch in [32,128,256] for precision in ['fp32','amp_fp16']]
    for config in TRAIN_CONFIGS:
        result['attempts'].append({'kind':'train','id':config['id'],'status':'started'})
        save()
        if time.monotonic()>=TRAIN_DEADLINE:
            row={'status':'budget_not_started','completed_warmup':0,'completed_measured':0}
        else:
            plan=bucket_plans[config['batch_pairs']] if config['bucketed'] else random8
            row=bench.benchmark_train(bench.MODEL_SPECS[config['model']],token_cache,plan,config['precision'],device,
                warmup=50,measured=200,fused=config['fused'],deadline=TRAIN_DEADLINE,profile=True)
        row.update(config)
        # Prefer a named model over duplicated specs in the summary table.
        result['train'].append(row)
        result['attempts'][-1]['status']=row['status']
        bench.write_json(RUN_DIR/'train'/f'{config["id"]}.json',row)
        save()
        print('TRAIN',json.dumps(row,ensure_ascii=False),flush=True)
    result['status']='training_grid_finished'
    save()
except BaseException as error:
    result['status']='training_failed_or_interrupted'
    result['training_error']=type(error).__name__+': '+str(error)[:400]
    save()
    raise

FUSED ADAMW AVAILABLE True
TRAIN {"model_spec": {"d_model": 256, "heads": 4, "encoder_layers": 4, "decoder_layers": 4, "swiglu_dim": 680}, "precision": "fp32", "batch_pairs": 8, "warmup_planned": 50, "measured_planned": 200, "fused_adamw": false, "status": "ok", "completed_warmup": 50, "completed_measured": 200, "parameters": 12086912, "sdpa": {"status": "observed", "operators": ["aten::_scaled_dot_product_efficient_attention", "aten::_scaled_dot_product_efficient_attention_backward", "aten::scaled_dot_product_attention"], "scope": "one warmup forward/backward for this shape/dtype, outside measured steps; backend may change with shapes"}, "warmup_seconds": 3.8944869729998572, "seconds": 9.099459402999855, "ms_per_step": 45.497297014999276, "pairs_per_second": 175.83462150207754, "useful_target_tokens_per_second": 11546.949697402993, "useful_source_tokens_per_second": 8494.900254680682, "useful_source_plus_target_tokens_per_second": 20041.849952083674, "counts": {"source_tokens": 77299,

## 6. Варианты greedy: сортировка + batches64/128/256/512 × FP32/FP16

FP16 здесь — `model.half()` на неизменных весах, без обучения и GradScaler. Изменение precision/batch может менять argmax, поэтому каждый вариант получает свой exec-match/долю token-identical. Контроль не меняется. План содержит9 вариантов всего,18 полных split-проходов (включая уже выполненный контроль).

Победитель: минимальный **полный wall test1919** среди завершённых вариантов с exec-match не ниже контроля более чем на0.5п.п. при denominator991. Это допускает максимум4 потерянных совпадения. Повторных замеров нет, влияние GPU clocks/шума не отделено. Соответствие лимиту180с показывается отдельно; формат JSONL — локальный, не подтверждённый формат бота.

In [7]:
if len(result['inference'])!=1:raise RuntimeError('Варианты уже запускались; используйте свежую сессию')
assert result.get('control_quality_gate_passed')
result['status']='inference_variants_running'
save()
try:
    projected_pair_seconds=1.25*(control['heldout']['seconds_full_wall']+control['test']['seconds_full_wall'])+15
    for variant in INFERENCE_VARIANTS[1:]:
        if time.monotonic()+projected_pair_seconds>=WORK_DEADLINE:
            row={'id':variant['id'],'config':variant,'status':'budget_not_started',
                'projection_seconds':projected_pair_seconds}
            result['inference'].append(row)
            bench.write_json(RUN_DIR/'inference'/f'{variant["id"]}.json',row)
            save()
        else:
            row=infer_variant(variant)
        print('INFERENCE',variant['id'],row['status'],flush=True)
    result['status']='benchmarks_finished'
    save()
except BaseException as error:
    result['status']='inference_failed_or_interrupted'
    result['inference_error']=type(error).__name__+': '+str(error)[:400]
    save()
    raise

EXEC sorted-fp32-b64 189 991
INFERENCE sorted-fp32-b64 ok
EXEC sorted-fp16-b64 191 991
INFERENCE sorted-fp16-b64 ok
EXEC sorted-fp32-b128 189 991
INFERENCE sorted-fp32-b128 ok
EXEC sorted-fp16-b128 191 991
INFERENCE sorted-fp16-b128 ok
EXEC sorted-fp32-b256 189 991
INFERENCE sorted-fp32-b256 ok
EXEC sorted-fp16-b256 190 991
INFERENCE sorted-fp16-b256 ok
EXEC sorted-fp32-b512 189 991
INFERENCE sorted-fp32-b512 ok
EXEC sorted-fp16-b512 190 991
INFERENCE sorted-fp16-b512 ok


## 7. Таблицы и решение по заранее заданным правилам

Решение относится к завершённым конфигурациям этого запуска. Оно не доказывает качество обучения больших batch/AMP/B/C и не заменяет длинный quality-контроль следующего опыта.

In [8]:
train_table=[]
train_control=next((r for r in result['train'] if r['id']=='A-fp32-b8-random-control' and r['status']=='ok'),None)
for row in result['train']:
    peak=max(row.get('peak_allocated_bytes') or 0,row.get('peak_reserved_bytes') or 0)
    display_row={k:row.get(k) for k in ['id','status','parameters','ms_per_step','pairs_per_second',
        'useful_target_tokens_per_second','useful_source_tokens_per_second','useful_source_plus_target_tokens_per_second',
        'source_padding_fraction','target_padding_fraction','total_padding_fraction','all_measured_losses_finite',
        'all_warmup_losses_finite','warmup_AMP_skipped_updates','measured_AMP_skipped_updates','completed_measured']}
    display_row['peak_GB']=peak/1e9
    display_row['peak_GiB']=peak/2**30
    display_row['target_speedup_vs_control']=row.get('useful_target_tokens_per_second',0)/train_control['useful_target_tokens_per_second'] if train_control and row['status']=='ok' else None
    display_row['pair_speedup_vs_control']=row.get('pairs_per_second',0)/train_control['pairs_per_second'] if train_control and row['status']=='ok' else None
    train_table.append(display_row)
train_frame=pd.DataFrame(train_table)
print('TRAIN TABLE')
display(train_frame)
train_frame.to_csv(RUN_DIR/'training-benchmark.csv',index=False)
inference_table=[]
for row in result['inference']:
    entry={'id':row['id'],'status':row['status']}
    for split in ['heldout','test']:
        summary=row.get(split,{})
        for key in ['seconds_historical_scope','seconds_full_wall','decode_steps_sum','decode_row_steps_sum',
                    'token_identical_fraction','peak_allocated_bytes','peak_reserved_bytes']:
            entry[f'{split}_{key}']=summary.get(key)
    score=row.get('heldout',{}).get('exec_match',{})
    entry['exec_matches']=score.get('matches');entry['exec_denominator']=score.get('denominator')
    entry['exec_accuracy']=score.get('accuracy')
    inference_table.append(entry)
infer_frame=pd.DataFrame(inference_table)
print('INFERENCE TABLE')
display(infer_frame)
infer_frame.to_csv(RUN_DIR/'inference-benchmark.csv',index=False)
train_best=bench.select_train(result['train'])
per_model={name:bench.select_train([r for r in result['train'] if r['model']==name]) for name in ['A','B','C']}
control_matches=control['heldout']['exec_match']['matches']
infer_best=bench.select_inference(result['inference'],control_matches)
decision={'train':train_best,'train_by_model':per_model,'inference':infer_best,
    'train_choice_rule':result['protocol']['train_selection'],
    'inference_choice_rule':result['protocol']['inference_selection'],
    'train_grid_has20_recorded_configs':len(result['train'])==20,
    'train_all20_finished_ok':len(result['train'])==20 and all(r['status']=='ok' for r in result['train']),
    'inference_has9_recorded_configs':len(result['inference'])==9,
    'inference_all9_finished_ok':len(result['inference'])==9 and all(r['status']=='ok' for r in result['inference']),
    'no_quality_training_conclusion':True}
if infer_best:
    decision['test_full_wall_below180_seconds']=infer_best['test']['seconds_full_wall']<180
    for split in ['test','heldout']:
        shutil.copyfile(RUN_DIR/'generations'/f'{infer_best["id"]}-{split}.json',RUN_DIR/f'best-{split}-generations.json')
        shutil.copyfile(RUN_DIR/'answers'/f'{infer_best["id"]}-{split}-answers.jsonl',RUN_DIR/f'best-{split}-answers.jsonl')
# Keep control+winner test generations; heldout generations retained for all completed variants.
for path in (RUN_DIR/'generations').glob('*-test.json'):
    keep_ids={control['id'],infer_best['id'] if infer_best else control['id']}
    if not any(path.name==f'{identifier}-test.json' for identifier in keep_ids):path.unlink()
result['decision']=decision
result['status']='completed_with_report'
bench.write_json(RUN_DIR/'decision.json',decision)
save()
print('TRAIN CHOICE:',train_best['id'] if train_best else 'Нет допустимого полного замера')
print('INFERENCE CHOICE:',infer_best['id'] if infer_best else 'Нет допустимого полного замера')
print('GRID STATUS',json.dumps({k:v for k,v in decision.items() if 'grid_' in k or '_all' in k or '_has' in k},ensure_ascii=False))
print('Elapsed minutes:',result['elapsed_seconds_since_first_cell']/60)
print('Этот запуск измеряет скорость и сохранность greedy-качества; качество будущего обучения с большим batch/AMP/B/C требует отдельного длинного контроля.')

TRAIN TABLE


,id,status,parameters,ms_per_step,pairs_per_second,useful_target_tokens_per_second,useful_source_tokens_per_second,useful_source_plus_target_tokens_per_second,source_padding_fraction,target_padding_fraction,total_padding_fraction,all_measured_losses_finite,all_warmup_losses_finite,warmup_AMP_skipped_updates,measured_AMP_skipped_updates,completed_measured,peak_GB,peak_GiB,target_speedup_vs_control,pair_speedup_vs_control
0,A-fp32-b8-random-control,ok,12086912.0,45.497297,175.834622,11546.949697,8494.900255,20041.849952,0.379503,0.377866,0.378561,True,True,0.0,0.0,200,0.981467,0.914062,1.000000,1.000000
1,A-amp_fp16-b8-random,ok,12086912.0,53.606230,149.236386,9800.260194,7209.889624,17010.149818,0.379503,0.377866,0.378561,True,True,0.0,0.0,200,0.694157,0.646484,0.848732,0.848732
2,A-fp32-b32-bucketed,ok,12086912.0,82.109212,389.724844,25730.486713,18800.874517,44531.361231,0.282709,0.201522,0.237938,True,True,0.0,0.0,200,2.810184,2.617188,2.228336,2.216428
3,A-amp_fp16-b32-bucketed,ok,12086912.0,53.391409,599.347356,39570.223534,28913.359299,68483.582833,0.282709,0.201522,0.237938,True,True,0.0,0.0,200,1.843397,1.716797,3.426898,3.408586
4,A-fp32-b128-bucketed,ok,12086912.0,324.945501,393.912209,26022.594501,19080.076463,45102.670965,0.354254,0.256185,0.301088,True,True,0.0,0.0,200,14.365491,13.378906,2.253634,2.240243
5,A-amp_fp16-b128-bucketed,ok,12086912.0,129.514183,988.308749,65289.567626,47871.089199,113160.656825,0.354254,0.256185,0.301088,True,True,0.0,0.0,200,8.980005,8.363281,5.654270,5.620672
6,A-fp32-b256-bucketed,oom,12086912.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN,0,14.833156,13.814453,NaN,NaN
7,A-amp_fp16-b256-bucketed,oom,12086912.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN,180,14.996734,13.966797,NaN,NaN
8,B-fp32-b32-bucketed,ok,31940096.0,199.309346,160.554438,10600.155192,7745.371856,18345.527049,0.282709,0.201522,0.237938,True,True,0.0,0.0,200,5.356126,4.988281,0.918005,0.913099
9,B-amp_fp16-b32-bucketed,ok,31940096.0,89.050903,359.345038,23724.745452,17335.309944,41060.055396,0.282709,0.201522,0.237938,True,True,0.0,0.0,200,3.309306,3.082031,2.054633,2.043653


INFERENCE TABLE


,id,status,heldout_seconds_historical_scope,heldout_seconds_full_wall,heldout_decode_steps_sum,heldout_decode_row_steps_sum,heldout_token_identical_fraction,heldout_peak_allocated_bytes,heldout_peak_reserved_bytes,test_seconds_historical_scope,test_seconds_full_wall,test_decode_steps_sum,test_decode_row_steps_sum,test_token_identical_fraction,test_peak_allocated_bytes,test_peak_reserved_bytes,exec_matches,exec_denominator,exec_accuracy
0,control-fp32-b32,ok,38.777916,39.185878,7137,228384,1.000000,232139264,291504128,70.876546,71.216653,13333,426557,1.000000,252803072,308281344,189,991,0.190716
1,sorted-fp32-b64,ok,18.818980,19.203037,3456,221184,1.000000,414571520,499122176,38.200721,38.592227,7104,454144,1.000000,452977664,547356672,189,991,0.190716
2,sorted-fp16-b64,ok,18.401623,19.056647,3488,223232,0.973633,216404992,266338304,40.065269,40.457821,7472,477696,0.968734,235627008,310378496,191,991,0.192735
3,sorted-fp32-b128,ok,15.144558,15.752749,2192,280576,1.000000,762611712,905969664,32.447146,32.815296,4688,599552,1.000000,841559552,1033895936,189,991,0.190716
4,sorted-fp16-b128,ok,12.441940,12.807423,2224,284672,0.966797,390217728,486539264,26.401021,26.781813,5072,648704,0.966128,430124032,549453824,191,991,0.192735
5,sorted-fp32-b256,ok,15.255332,15.643068,1232,315392,1.000000,1456332288,1889533952,38.241143,38.901882,3504,830976,1.000000,1333529088,1539309568,189,991,0.190716
6,sorted-fp16-b256,ok,6.630063,7.000494,1232,315392,0.970703,738629120,979369984,20.677704,21.377823,3888,929280,0.964044,676381184,794820608,190,991,0.191726
7,sorted-fp32-b512,ok,15.995139,16.344981,624,319488,1.000000,2846394880,3776970752,39.586603,39.961373,1840,876032,1.000000,2526134784,3531603968,189,991,0.190716
8,sorted-fp16-b512,ok,5.525519,5.885514,624,319488,0.974609,1433879040,1793064960,15.186943,15.566698,2048,982528,0.969255,1274182144,1818230784,190,991,0.191726


TRAIN CHOICE: A-amp_fp16-b128-bucketed
INFERENCE CHOICE: sorted-fp16-b512
GRID STATUS {"train_grid_has20_recorded_configs": true, "train_all20_finished_ok": false, "inference_has9_recorded_configs": true, "inference_all9_finished_ok": true}
Elapsed minutes: 45.48938487905001
Этот запуск измеряет скорость и сохранность greedy-качества; качество будущего обучения с большим batch/AMP/B/C требует отдельного длинного контроля.


## 8. Один ZIP — выполнять также после прерывания

Ячейка независима от модулей/успешного бенчмарка. Упаковывает существующие результаты, отмечает незавершённый статус и добавляет SHA256-manifest. PT/данные/кеш/загруженный исходный ZIP исключены. При прерывании внутри split его завершённые файлы и последние сохранённые конфигурации останутся в архиве.

In [9]:
import json,hashlib,zipfile,time
from pathlib import Path
from google.colab import files
folder=Path('hw02-bench23-results')
folder.mkdir(exist_ok=True)
result_path=folder/'results.json'
if result_path.exists():
    payload=json.loads(result_path.read_text(encoding='utf-8'))
else:
    payload={'experiment':23,'status':'partial_before_setup'}
if payload.get('status')!='completed_with_report':
    payload['partial_archive']=True
    payload['interrupted_or_unfinished_phase']=payload.get('status')
if '_BENCH23_STARTED' in globals():
    payload['elapsed_seconds_at_pack']=time.monotonic()-_BENCH23_STARTED
    payload['within60min_at_pack_start']=payload['elapsed_seconds_at_pack']<=3600
result_path.write_text(json.dumps(payload,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
entries=[]
for path in sorted(folder.rglob('*')):
    if not path.is_file() or path.name=='manifest.json' or path.suffix in {'.pt','.parquet','.zip','.tmp'}:
        continue
    raw=path.read_bytes()
    entries.append({'file':str(path.relative_to(folder)),'bytes':len(raw),'sha256':hashlib.sha256(raw).hexdigest()})
(folder/'manifest.json').write_text(json.dumps({'experiment':23,'files':entries},ensure_ascii=False,indent=2),encoding='utf-8')
archive=Path('hw02-throughput-batch-amp-bucketing-and-batched-inference-results.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
    for path in sorted(folder.rglob('*')):
        if path.is_file() and path.suffix not in {'.pt','.parquet','.zip','.tmp'}:
            z.write(path,arcname=str(path.relative_to(folder)))
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None,'ZIP CRC failed'
    manifest=json.loads(z.read('manifest.json'))
    for entry in manifest['files']:
        raw=z.read(entry['file'])
        assert len(raw)==entry['bytes'] and hashlib.sha256(raw).hexdigest()==entry['sha256']
print('ARCHIVE',archive.name,'bytes',archive.stat().st_size,'status',payload.get('status'))
files.download(str(archive))

ARCHIVE hw02-throughput-batch-amp-bucketing-and-batched-inference-results.zip bytes 8557040 status completed_with_report


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>